In [0]:
# Databricks notebook source
# Run the shared authentication context notebook
%run /Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb

In [0]:
# Databricks notebook source
# ==============================================================================
# name        : layer_reconciliation_report
# purpose     : Audits data across Bronze and Silver layers for all 14 tables.
#               Identifies why fact tables like health_snapshots or activity_snapshots
#               might drop rows during cleaning transformations.
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StructType
import json

# 1. Pipeline Metadata Configuration
TABLES_METADATA = [
    {"table_name": "owners", "table_type": "dimension", "pk": ["owner_id"]},
    {"table_name": "pets", "table_type": "dimension", "pk": ["pet_id"]},
    {"table_name": "products", "table_type": "dimension", "pk": ["product_id"]},
    {"table_name": "firmware", "table_type": "dimension", "pk": ["firmware_id"]},
    {"table_name": "batches", "table_type": "dimension", "pk": ["batch_id"]},
    {"table_name": "devices", "table_type": "dimension", "pk": ["device_id"]},
    {"table_name": "sales", "table_type": "fact", "pk": ["sale_id"]},
    {"table_name": "device_snapshots", "table_type": "fact", "pk": ["device_id", "snapshot_timestamp"]},
    {"table_name": "activity_snapshots", "table_type": "fact", "pk": ["device_id", "snapshot_timestamp"]},
    {"table_name": "health_snapshots", "table_type": "fact", "pk": ["device_id", "snapshot_timestamp"]},
    {"table_name": "feeding_events", "table_type": "fact", "pk": ["feed_event_id"]},
    {"table_name": "hydration_events", "table_type": "fact", "pk": ["water_event_id"]},
    {"table_name": "device_failures", "table_type": "fact", "pk": ["failure_id"]},
    {"table_name": "firmware_deployments", "table_type": "fact", "pk": ["deployment_id"]}
]

def check_table_reconciliation(meta):
    t_name = meta["table_name"]
    t_type = meta["table_type"]
    pks = meta["pk"]
    
    bronze_path = f"abfss://bronze@petiot.dfs.core.windows.net/{t_name}"
    silver_path = f"abfss://silver@petiot.dfs.core.windows.net/{t_name}"
    
    report = {
        "table_name": t_name,
        "table_type": t_type,
        "bronze_exists": False,
        "silver_exists": False,
        "bronze_total_rows": 0,
        "silver_total_rows": 0,
        "status_summary": "OK",
        "anomalies_detected": []
    }
    
    # ---- Step 1: Read Bronze Layer ----
    try:
        df_bronze = spark.read.format("delta").load(bronze_path)
        report["bronze_exists"] = True
        report["bronze_total_rows"] = df_bronze.count()
    except Exception as e:
        report["status_summary"] = "BRONZE_MISSING"
        report["anomalies_detected"].append(f"Cannot read Bronze Delta path: {str(e)[:100]}")
        return report

    # ---- Step 2: Read Silver Layer ----
    try:
        df_silver = spark.read.format("delta").load(silver_path)
        report["silver_exists"] = True
        report["silver_total_rows"] = df_silver.count()
    except Exception as e:
        report["status_summary"] = "SILVER_MISSING"
        report["anomalies_detected"].append(f"Cannot read Silver Delta path: {str(e)[:100]}")
        # Continue evaluation so we can inspect what's sitting raw in Bronze
        df_silver = None

    # ---- Step 3: Deep Anomaly & Drop Detection Rules ----
    if report["bronze_total_rows"] > 0 and (df_silver is None or report["silver_total_rows"] == 0):
        report["status_summary"] = "CRITICAL_0_ROWS_IN_SILVER"
        
        # Check A: Inspect _load_date breakdown in Bronze
        if "_load_date" in df_bronze.columns:
            distinct_dates = [r["_load_date"] for r in df_bronze.groupBy("_load_date").count().collect()]
            report["anomalies_detected"].append(f"Bronze has data for dates: {distinct_dates}")
        else:
            report["anomalies_detected"].append("Bronze lacks operational '_load_date' partition lineage column.")
            
        # Check B: Test table-specific filtering hazards from cleaner.py
        if t_name == "health_snapshots":
            # Count rows where vital statistics are completely missing or blank
            vitals_check = df_bronze.filter(
                (F.col("heart_rate").isNull() | (F.col("heart_rate") == "")) &
                (F.col("pulse_rate").isNull() | (F.col("pulse_rate") == "")) &
                (F.col("body_temperature").isNull() | (F.col("body_temperature") == ""))
            ).count()
            if vitals_check > 0:
                report["anomalies_detected"].append(
                    f"Found {vitals_check} Bronze rows where heart_rate, pulse_rate, and body_temperature are simultaneously blank/null. Cleaner drops these entirely."
                )
                
        if t_name == "activity_snapshots":
            # Check for corrupt keys collapsing under dropDuplicates
            null_keys = df_bronze.filter((F.col("device_id").isNull()) | (F.col("device_id") == "") | (F.col("snapshot_timestamp").isNull()) | (F.col("snapshot_timestamp") == "")).count()
            if null_keys > 0:
                report["anomalies_detected"].append(
                    f"Found {null_keys} rows with empty device_id or snapshot_timestamp. dropDuplicates() will collapse these down to 1 row."
                )

    # Check C: Duplicate rate evaluation
    if df_silver is not None and report["silver_total_rows"] > 0:
        # Cross check if duplicates in primary keys exist in Bronze causing volume discrepancy
        pk_exprs = [F.col(c) for c in pks]
        unique_bronze_pks = df_bronze.select(pk_exprs).distinct().count()
        if unique_bronze_pks < report["bronze_total_rows"]:
            dedup_diff = report["bronze_total_rows"] - unique_bronze_pks
            report["anomalies_detected"].append(f"Bronze has {dedup_diff} duplicate primary key records that Silver filtered out.")
            
    if len(report["anomalies_detected"]) == 0:
        report["anomalies_detected"].append("No discrepancies found.")
        
    return report

# 2. Loop through all tables and compile data
reconciliation_results = []
print("Starting global layer audit across Lakehouse containers...")

for table_meta in TABLES_METADATA:
    print(f"-> Auditing reconciliation status for table: {table_meta['table_name']}")
    res = check_table_reconciliation(table_meta)
    reconciliation_results.append(res)

# 3. Format into a structured clean DataFrame Report
schema = StructType()
report_df = spark.createDataFrame(reconciliation_results)

# Explode anomalies list to make it perfectly readable in output grid views
final_report_df = report_df.withColumn("anomalies_detected", F.concat_ws(" | ", F.col("anomalies_detected")))

# 4. Display Results
print("\n" + "="*80)
print("             DELTA LAKE LAYER RECONCILIATION SUMMARY REPORT")
print("="*80)
final_report_df.select(
    "table_name", 
    "table_type", 
    "bronze_total_rows", 
    "silver_total_rows", 
    "status_summary", 
    "anomalies_detected"
).show(20, truncate=False)

# Display interactive table if running in Databricks UI workspace
display(final_report_df)

In [0]:
# Databricks notebook source
# ==============================================================================
# DIAGNOSTIC NOTEBOOK: Action Plan Verification (Fixed for All Spark Versions)
# Layer     : Silver Layer Debugging
# Purpose   : Profiling raw strings in Bronze to verify why dropna() wipes out rows
#             Uses expr() to guarantee compatibility across all DBR runtimes.
# ==============================================================================

from pyspark.sql import functions as F

# Storage account path config
STORAGE_ACCOUNT = "petiot"
HYDRO_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/hydration_events"
HEALTH_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/health_snapshots"

print("="*80)
print(" 1. DIAGNOSING HYDRATION EVENTS (water_consumed_ml)")
print("="*80)

try:
    df_hydro_raw = spark.read.format("delta").load(HYDRO_PATH)
    
    # 1. Analyze the exact distribution of character string values in the raw layer
    print("[INFO] Top 10 raw text string values in Bronze 'water_consumed_ml' column:")
    df_hydro_raw.groupBy("water_consumed_ml").count().orderBy(F.col("count").desc()).show(10, truncate=False)
    
    # 2. Simulate cleaning rules using expr() instead of py wrapper imports
    total_bronze_hydro = df_hydro_raw.count()
    simulated_cast_hydro = df_hydro_raw.withColumn(
        "water_consumed_ml_cast", 
        F.expr("try_cast(water_consumed_ml AS double)")
    )
    
    null_after_cast_hydro = simulated_cast_hydro.filter(F.col("water_consumed_ml_cast").isNull()).count()
    saved_without_dropna_hydro = simulated_cast_hydro.count()
    saved_with_dropna_hydro = simulated_cast_hydro.dropna(subset=["water_consumed_ml_cast"]).count()
    
    print(f"\n--- Simulation Reconciliation ---")
    print(f"Total Raw Rows in Bronze      : {total_bronze_hydro:,}")
    print(f"Rows Evaluated to NULL by cast: {null_after_cast_hydro:,} ({ (null_after_cast_hydro/total_bronze_hydro)*100 if total_bronze_hydro > 0 else 0:.1f}%)")
    print(f"Rows Saved WITHOUT dropna()   : {saved_without_dropna_hydro:,} -> [Expected if dropna removed]")
    print(f"Rows Saved WITH dropna()      : {saved_with_dropna_hydro:,} -> [Current Behavior (0 Rows)]")
    
except Exception as e:
    print(f"[ERROR] Failed checking hydration path: {e}")


print("\n" + "="*80)
print(" 2. DIAGNOSING HEALTH SNAPSHOTS (vitals elements)")
print("="*80)

try:
    df_health_raw = spark.read.format("delta").load(HEALTH_PATH)
    
    # 1. Profile what the character elements look like in the raw entries
    print("[INFO] Top 10 raw string patterns inside health snapshot vitals:")
    df_health_raw.groupBy("heart_rate", "pulse_rate", "body_temperature") \
                 .count() \
                 .orderBy(F.col("count").desc()) \
                 .show(10, truncate=False)
    
    # 2. Simulate health transformations using native SQL try_cast expressions
    total_bronze_health = df_health_raw.count()
    simulated_cast_health = df_health_raw \
        .withColumn("hr_cast", F.expr("try_cast(heart_rate AS double)")) \
        .withColumn("pr_cast", F.expr("try_cast(pulse_rate AS double)")) \
        .withColumn("bt_cast", F.expr("try_cast(body_temperature AS double)"))
        
    all_vitals_null_count = simulated_cast_health.filter(
        F.col("hr_cast").isNull() & F.col("pr_cast").isNull() & F.col("bt_cast").isNull()
    ).count()
    
    saved_with_dropna_health = simulated_cast_health.dropna(subset=["hr_cast", "pr_cast", "bt_cast"], how="all").count()
    
    print(f"\n--- Simulation Reconciliation ---")
    print(f"Total Raw Rows in Bronze      : {total_bronze_health:,}")
    print(f"Rows where ALL vitals are NULL: {all_vitals_null_count:,} ({ (all_vitals_null_count/total_bronze_health)*100 if total_bronze_health > 0 else 0:.1f}%)")
    print(f"Rows Saved WITHOUT dropna()   : {total_bronze_health:,} -> [Expected if dropna removed]")
    print(f"Rows Saved WITH dropna()      : {saved_with_dropna_health:,} -> [Current Behavior (0 Rows)]")

except Exception as e:
    print(f"[ERROR] Failed checking health path: {e}")

In [0]:
# Check the actual partition string shapes in your Bronze tables
print("--- Health Snapshots Bronze Partitions ---")
spark.read.format("delta").load("abfss://bronze@petiot.dfs.core.windows.net/health_snapshots") \
     .groupBy("_load_date").count().show(5, truncate=False)

print("--- Hydration Events Bronze Partitions ---")
spark.read.format("delta").load("abfss://bronze@petiot.dfs.core.windows.net/hydration_events") \
     .groupBy("_load_date").count().show(5, truncate=False)